## Skin Acne Classification - MILD/MODERATE/SEVERE/VERY SEVERE
* Dataset - Kaggle ACNE04
* Link for the dataset - https://www.kaggle.com/datasets/manuelhettich/acne04
* Model - Mk1
* Date - 07/03/2026
* ResNet50 is used in this - accuracy on validation - 77%

File structure redesign
* The file structure is not desired so we will be changing the file structure.
* The function will create the following file structure - data -> train, val

In [1]:
import os
import shutil
import random

def prepare_dataset_structure(source_base_path, target_base_path, split_ratio=0.8):
    """
    Parses the native ACNE04 Kaggle dataset structure and converts it into a 
    PyTorch ImageFolder-compatible architecture with train and val splits.
    
    Args:
        source_base_path (str): The root path containing the 'acne_1024' folder.
        target_base_path (str): The output directory (e.g., in /kaggle/working/).
        split_ratio (float): Percentage of data to be used for training (default 80%).
    """
    # Map the target class names to the exact folder names from your image
    source_folders = {
        'level0': 'acne0_1024',
        'level1': 'acne1_1024',
        'level2': 'acne2_1024',
        'level3': 'acne3_1024'
    }

    print(f"Creating new dataset structure at: {target_base_path}")
    
    # Create target directories for both train and val sets
    for split in ['train', 'val']:
        for class_name in source_folders.keys():
            os.makedirs(os.path.join(target_base_path, split, class_name), exist_ok=True)

    # Set a fixed random seed to ensure reproducibility across different runs
    random.seed(42)

    # Process each source folder
    for class_label, folder_name in source_folders.items():
        source_dir = os.path.join(source_base_path, folder_name)

        if not os.path.exists(source_dir):
            print(f"Warning: Source directory not found - {source_dir}")
            continue

        # Get all valid image files in the directory
        files = [f for f in os.listdir(source_dir) if os.path.isfile(os.path.join(source_dir, f))]
        
        # Shuffle the files to ensure random distribution between train and val
        random.shuffle(files)

        # Calculate the index to split the dataset
        split_idx = int(len(files) * split_ratio)
        train_files = files[:split_idx]
        val_files = files[split_idx:]

        # Copy files to the corresponding 'train' directory
        for f in train_files:
            src_path = os.path.join(source_dir, f)
            dst_path = os.path.join(target_base_path, 'train', class_label, f)
            shutil.copy2(src_path, dst_path)

        # Copy files to the corresponding 'val' directory
        for f in val_files:
            src_path = os.path.join(source_dir, f)
            dst_path = os.path.join(target_base_path, 'val', class_label, f)
            shutil.copy2(src_path, dst_path)

        print(f"Processed {class_label} ({folder_name}): Copied {len(train_files)} to train, {len(val_files)} to val.")

# ==========================================
# Example usage for your Kaggle environment
# ==========================================
# Update the 'source_path' to match your exact Kaggle input path
source_path = '/kaggle/input/datasets/manuelhettich/acne04/acne_1024' 
target_path = '/kaggle/working/data'

prepare_dataset_structure(source_path, target_path, split_ratio=0.8)

Creating new dataset structure at: /kaggle/working/data
Processed level0 (acne0_1024): Copied 386 to train, 97 to val.
Processed level1 (acne1_1024): Copied 498 to train, 125 to val.
Processed level2 (acne2_1024): Copied 140 to train, 35 to val.
Processed level3 (acne3_1024): Copied 77 to train, 20 to val.


In [2]:
"""
=============================================================================
Acne Severity Classification: Phase 1 - Training Pipeline
Framework: PyTorch
Target Classes: 0: Mild Acne, 1: Moderate Acne, 2: Severe Acne, 3: Very Severe Acne
Environment: Kaggle Notebooks (GPU Enabled)
=============================================================================
This module handles dataset parsing, dynamic augmentation, transfer learning
initialization, gradient optimization, and model serialization.
"""

import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import numpy as np
import warnings

# Suppress non-critical warnings for cleaner Kaggle logs
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# 1. Global Configuration and Hyperparameters
# ---------------------------------------------------------------------------
CONFIG = {
    # Kaggle dataset path containing the 'train' and 'val' split folders
    # Directories inside should be: level0, level1, level2, level3
    "data_dir": "/kaggle/working/data", 
    "batch_size": 32,            # Number of images processed per forward pass
    "num_epochs": 25,            # Total full passes over the training dataset
    "learning_rate": 1e-4,       # Initial step size for the optimizer
    "weight_decay": 1e-4,        # L2 regularization penalty to prevent overfitting
    "num_classes": 4,            # Target taxonomic classes
    "model_save_path": "/kaggle/working/acne_resnet50_best.pth", # Output artifact
    "label_smoothing": 0.1       # Softens one-hot labels to manage grading ambiguity
}

# Hardware abstraction: Utilize CUDA cores if a Kaggle GPU is active
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Initializing pipeline on hardware accelerator: {device}")

# ---------------------------------------------------------------------------
# 2. Data Transformation and Tensor Standardization
# ---------------------------------------------------------------------------
# Pre-trained models expect specific pixel normalization based on ImageNet.
# Augmentations are applied exclusively to the training subset to induce robustness.
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet mean
                             std=[0.229, 0.224, 0.225])   # ImageNet std
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ]),
}

# ---------------------------------------------------------------------------
# 3. Dataset Parsing and DataLoader Instantiation
# ---------------------------------------------------------------------------
def build_dataloaders(data_dir, batch_size):
    """
    Constructs iterable PyTorch DataLoaders. The ImageFolder class dynamically
    maps subdirectories to integer labels based on alphabetical sorting.
    """
    if not os.path.exists(data_dir):
        raise FileNotFoundError(f"Critical Error: Dataset directory {data_dir} not found.")

    image_datasets = {
        x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
        for x in ['train', 'val']
    }
    
    # num_workers enables multi-processed data fetching to prevent GPU starvation
    dataloaders = {
        x: DataLoader(image_datasets[x], batch_size=batch_size,
                      shuffle=(x == 'train'), num_workers=2)
        for x in ['train', 'val']
    }
    
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
    class_names = image_datasets['train'].classes
    
    print(f"[INFO] Taxonomy mapped to indices: {class_names}")
    print(f"[INFO] Ingested {dataset_sizes['train']} training and {dataset_sizes['val']} validation tensors.")
    
    return dataloaders, dataset_sizes, class_names

# ---------------------------------------------------------------------------
# 4. Transfer Learning Model Architecture
# ---------------------------------------------------------------------------
def initialize_model(num_classes):
    """
    Instantiates a ResNet-50 Convolutional Neural Network pre-trained on ImageNet.
    Truncates the terminal 1000-class classification head and replaces it with
    a 4-class linear projection mapping to our specific acne severity taxonomy.
    """
    print("[INFO] Downloading and initializing pre-trained ResNet-50 architecture...")
    model = models.resnet50(pretrained=True)
    
    # Isolate the input dimensionality of the final linear layer
    num_ftrs = model.fc.in_features
    
    # Re-initialize the terminal layer for the specific taxonomic task
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    # Migrate the entire computational graph to the specified device (GPU/CPU)
    model = model.to(device)
    return model

# ---------------------------------------------------------------------------
# 5. Core Algorithmic Optimization Engine
# ---------------------------------------------------------------------------
def train_model(model, dataloaders, dataset_sizes, criterion, optimizer, scheduler, num_epochs):
    """
    Executes the gradient descent lifecycle. Iterates through epochs, alternating
    between forward/backward propagation (training) and deterministic evaluation (validation).
    Implements model checkpointing by preserving the weights with the highest validation accuracy.
    """
    since = time.time()
    
    # Initialize a deep copy object to store the optimal parameter state
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 20)

        # Alternating phases dictate whether layers like Dropout and BatchNorm update
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Enable computational graph recording and regularization
            else:
                model.eval()   # Disable regularization, freeze network for pure inference

            running_loss = 0.0
            running_corrects = 0

            # Batch processing iteration
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Reset gradients; PyTorch natively accumulates them across backward passes
                optimizer.zero_grad()

                # Context manager: calculate gradients ONLY if optimizing
                with torch.set_grad_enabled(phase == 'train'):
                    # Forward pass: Project inputs through the network layers
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    
                    # Calculate divergence between prediction probabilities and true labels
                    loss = criterion(outputs, labels)

                    # Backward pass: Calculate partial derivatives and adjust weights
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Accumulate statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Decay the learning rate based on the defined schedule
            if phase == 'train':
                scheduler.step()

            # Aggregate epoch-level metrics
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'[{phase.upper()}] Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}')

            # Model Checkpointing: Preserve weights if validation accuracy surpasses the historic best
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                print(f"   --> Validation accuracy improved. Caching model state.")

    time_elapsed = time.time() - since
    print(f'\n[INFO] Optimization sequence concluded in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'[INFO] Peak Validation Accuracy Achieved: {best_acc:.4f}')

    # Inject the optimal historical weights back into the network shell
    model.load_state_dict(best_model_wts)
    return model

# ---------------------------------------------------------------------------
# 6. Primary Execution Routine
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    try:
        # 1. Prepare Data Pipelines
        dataloaders, dataset_sizes, class_names = build_dataloaders(CONFIG["data_dir"], CONFIG["batch_size"])
        
        # 2. Architect the Network
        model_ft = initialize_model(CONFIG["num_classes"])
        
        # 3. Define the Loss Protocol
        # CrossEntropyLoss inherently processes raw logits and applies LogSoftmax mathematically
        # label_smoothing mitigates overconfidence, crucial for ambiguous severity boundaries
        criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
        
        # 4. Define the Optimizer
        # AdamW separates weight decay from the gradient update, providing superior generalization
        optimizer_ft = optim.AdamW(model_ft.parameters(), 
                                   lr=CONFIG["learning_rate"], 
                                   weight_decay=CONFIG["weight_decay"])
        
        # 5. Define Learning Rate Scheduler
        # Decreases the learning rate dynamically to allow fine-grained convergence in later epochs
        exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)
        
        # 6. Execute the Training Sequence
        print("\n[INFO] Engaging gradient descent optimization loop...")
        optimized_model = train_model(model_ft, dataloaders, dataset_sizes, criterion, 
                                      optimizer_ft, exp_lr_scheduler, num_epochs=CONFIG["num_epochs"])
        
        # 7. Serialize the Optimized Architecture
        # The state_dict contains all learned parameter tensors (weights and biases)
        torch.save(optimized_model.state_dict(), CONFIG["model_save_path"])
        print(f"\n Production model artifact serialized and stored at: {CONFIG['model_save_path']}")
        print("          This.pth file can now be exported for independent inference applications.")
        
    except FileNotFoundError as e:
        print(e)
        print("Please structure your Kaggle input directory utilizing the PyTorch ImageFolder paradigm.")

[INFO] Initializing pipeline on hardware accelerator: cuda:0
[INFO] Taxonomy mapped to indices: ['level0', 'level1', 'level2', 'level3']
[INFO] Ingested 1100 training and 277 validation tensors.
[INFO] Downloading and initializing pre-trained ResNet-50 architecture...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 196MB/s] 



[INFO] Engaging gradient descent optimization loop...

Epoch 1/25
--------------------
[TRAIN] Loss: 1.0121 | Accuracy: 0.6000
[VAL] Loss: 0.9088 | Accuracy: 0.6751
   --> Validation accuracy improved. Caching model state.

Epoch 2/25
--------------------
[TRAIN] Loss: 0.5946 | Accuracy: 0.8900
[VAL] Loss: 0.9129 | Accuracy: 0.6931
   --> Validation accuracy improved. Caching model state.

Epoch 3/25
--------------------
[TRAIN] Loss: 0.4675 | Accuracy: 0.9645
[VAL] Loss: 0.8486 | Accuracy: 0.7365
   --> Validation accuracy improved. Caching model state.

Epoch 4/25
--------------------
[TRAIN] Loss: 0.4291 | Accuracy: 0.9809
[VAL] Loss: 0.8172 | Accuracy: 0.7292

Epoch 5/25
--------------------
[TRAIN] Loss: 0.4002 | Accuracy: 0.9900
[VAL] Loss: 0.8192 | Accuracy: 0.7220

Epoch 6/25
--------------------
[TRAIN] Loss: 0.3969 | Accuracy: 0.9882
[VAL] Loss: 0.8860 | Accuracy: 0.7076

Epoch 7/25
--------------------
[TRAIN] Loss: 0.3880 | Accuracy: 0.9891
[VAL] Loss: 0.8315 | Accuracy: 0